# Graph Representations - Worked Examples

This notebook covers 12 practical implementations of different graph representation formats used in machine learning and graph neural networks.

**Topics Covered:**
1. Adjacency Matrix
2. Adjacency List
3. Incidence Matrix
4. Edge List
5. Sparse Representations (COO Format)
6. CSR/CSC Formats
7. Memory Comparison
8. Conversion Between Formats
9. Weighted Graph Representations
10. Directed Graph Representations
11. Multigraph Representations
12. Choosing the Right Representation

**Libraries Used:** `networkx`, `numpy`, `scipy.sparse`

In [ ]:
import numpy as np
import networkx as nx
from scipy import sparse
from collections import defaultdict
from typing import List, Tuple, Dict, Set, Optional, Any
import sys

---
## Example 1: Adjacency Matrix Representation

The **adjacency matrix** is the most fundamental graph representation. For a graph with $n$ vertices:
- $A[i,j] = 1$ if there is an edge from vertex $i$ to vertex $j$
- $A[i,j] = 0$ otherwise

**Properties:**
- Space complexity: $O(n^2)$
- Edge lookup: $O(1)$
- Finding neighbors: $O(n)$
- Matrix powers $A^k$ count paths of length $k$

In [ ]:
class AdjacencyMatrix:
    """Graph representation using adjacency matrix."""
    
    def __init__(self, n: int, directed: bool = False):
        self.n = n
        self.directed = directed
        self.matrix = np.zeros((n, n), dtype=np.float32)
    
    def add_edge(self, u: int, v: int, weight: float = 1.0):
        self.matrix[u, v] = weight
        if not self.directed:
            self.matrix[v, u] = weight
    
    def has_edge(self, u: int, v: int) -> bool:
        return self.matrix[u, v] != 0
    
    def get_neighbors(self, v: int) -> List[int]:
        return list(np.where(self.matrix[v] != 0)[0])
    
    def degree(self, v: int) -> int:
        return int(np.sum(self.matrix[v] != 0))
    
    def num_edges(self) -> int:
        count = np.sum(self.matrix != 0)
        return int(count if self.directed else count // 2)

# Create a cycle graph: 0 --- 1
#                       |     |
#                       3 --- 2
g = AdjacencyMatrix(4)
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
for u, v in edges:
    g.add_edge(u, v)

print("Graph: 0-1-2-3-0 (cycle)")
print(f"\nAdjacency Matrix:\n{g.matrix.astype(int)}")
print(f"\nEdge (0,1) exists: {g.has_edge(0, 1)}")
print(f"Edge (0,2) exists: {g.has_edge(0, 2)}")
print(f"Neighbors of 1: {g.get_neighbors(1)}")
print(f"Degree of vertex 2: {g.degree(2)}")

# Matrix powers for path counting
A = g.matrix
A2 = A @ A
print(f"\nA² (counts 2-hop paths):\n{A2.astype(int)}")
print(f"Number of 2-hop paths from 0 to 2: {int(A2[0, 2])}")

---
## Example 2: Adjacency List Representation

The **adjacency list** stores, for each vertex, a list of its neighbors.

**Properties:**
- Space complexity: $O(n + m)$ where $m$ is number of edges
- Edge lookup: $O(\text{degree})$ with list, $O(1)$ with set
- Finding neighbors: $O(1)$ access to list
- More memory-efficient for sparse graphs

In [ ]:
class AdjacencyList:
    """Graph representation using adjacency list."""
    
    def __init__(self, directed: bool = False):
        self.adj = defaultdict(list)
        self.directed = directed
        self._edges = 0
    
    def add_vertex(self, v: int):
        if v not in self.adj:
            self.adj[v] = []
    
    def add_edge(self, u: int, v: int, weight: float = 1.0):
        self.adj[u].append((v, weight))
        if not self.directed:
            self.adj[v].append((u, weight))
        self._edges += 1
    
    def has_edge(self, u: int, v: int) -> bool:
        return any(neighbor == v for neighbor, _ in self.adj[u])
    
    def get_neighbors(self, v: int) -> List[int]:
        return [neighbor for neighbor, _ in self.adj[v]]
    
    def get_weighted_neighbors(self, v: int) -> List[Tuple[int, float]]:
        return self.adj[v].copy()
    
    def degree(self, v: int) -> int:
        return len(self.adj[v])
    
    def vertices(self) -> List[int]:
        return list(self.adj.keys())

# Create weighted graph
g = AdjacencyList()
weighted_edges = [
    (0, 1, 1.0), (0, 2, 4.0), (1, 2, 2.0),
    (1, 3, 5.0), (2, 3, 1.0)
]

for u, v, w in weighted_edges:
    g.add_edge(u, v, w)

print("Weighted graph adjacency list:")
for v in sorted(g.vertices()):
    neighbors = g.get_weighted_neighbors(v)
    print(f"  {v}: {neighbors}")

print(f"\nNeighbors of vertex 1: {g.get_neighbors(1)}")
print(f"Degree of vertex 2: {g.degree(2)}")

---
## Example 3: Incidence Matrix Representation

The **incidence matrix** $B$ relates vertices to edges:
- For undirected graphs: $B[v,e] = 1$ if vertex $v$ is an endpoint of edge $e$
- For directed graphs: $B[v,e] = -1$ (source), $+1$ (target)

**Key property:** The graph Laplacian $L = B B^T$ for directed incidence matrix.

In [ ]:
class IncidenceGraph:
    """Graph using incidence matrix representation."""
    
    def __init__(self, n: int):
        self.n = n
        self.edges = []  # List of (u, v) pairs
    
    def add_edge(self, u: int, v: int):
        self.edges.append((u, v))
    
    def get_incidence_matrix_undirected(self) -> np.ndarray:
        """B[v, e] = 1 if vertex v is endpoint of edge e."""
        m = len(self.edges)
        B = np.zeros((self.n, m))
        for e, (u, v) in enumerate(self.edges):
            B[u, e] = 1
            B[v, e] = 1
        return B
    
    def get_incidence_matrix_directed(self) -> np.ndarray:
        """B[v, e] = -1 (tail/source), +1 (head/target)."""
        m = len(self.edges)
        B = np.zeros((self.n, m))
        for e, (u, v) in enumerate(self.edges):
            B[u, e] = -1  # Tail (source)
            B[v, e] = 1   # Head (target)
        return B
    
    def laplacian_from_incidence(self) -> np.ndarray:
        """L = B @ B^T for directed incidence matrix."""
        B = self.get_incidence_matrix_directed()
        return B @ B.T

# Create a square graph
g = IncidenceGraph(4)
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
for u, v in edges:
    g.add_edge(u, v)

print("Graph edges: 0→1, 1→2, 2→3, 3→0")

B_undir = g.get_incidence_matrix_undirected()
print(f"\nUndirected incidence matrix (rows=vertices, cols=edges):\n{B_undir.astype(int)}")

B_dir = g.get_incidence_matrix_directed()
print(f"\nDirected incidence matrix:\n{B_dir.astype(int)}")

# Laplacian from incidence
L = g.laplacian_from_incidence()
print(f"\nLaplacian L = B @ B^T:\n{L.astype(int)}")

# Verify: L = D - A
A = np.array([[0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0]])
D = np.diag(np.sum(A, axis=1))
L_verify = D - A
print(f"\nVerification L = D - A:\n{L_verify.astype(int)}")

---
## Example 4: Edge List Representation

The **edge list** is the simplest representation: just a list of $(u, v)$ pairs.

**Properties:**
- Space complexity: $O(m)$
- Edge lookup: $O(m)$ - slow!
- Simple but inefficient for most operations
- Good for input/output and edge-centric algorithms

In [ ]:
class EdgeList:
    """Graph using edge list representation."""
    
    def __init__(self, directed: bool = False):
        self.edges: List[Tuple[int, int, float]] = []
        self.directed = directed
    
    def add_edge(self, u: int, v: int, weight: float = 1.0):
        self.edges.append((u, v, weight))
    
    def has_edge(self, u: int, v: int) -> bool:
        for src, dst, _ in self.edges:
            if src == u and dst == v:
                return True
            if not self.directed and src == v and dst == u:
                return True
        return False
    
    def get_neighbors(self, v: int) -> List[int]:
        neighbors = []
        for src, dst, _ in self.edges:
            if src == v:
                neighbors.append(dst)
            elif not self.directed and dst == v:
                neighbors.append(src)
        return neighbors
    
    def vertices(self) -> Set[int]:
        verts = set()
        for u, v, _ in self.edges:
            verts.add(u)
            verts.add(v)
        return verts
    
    def to_adjacency_matrix(self) -> np.ndarray:
        n = max(max(u, v) for u, v, _ in self.edges) + 1
        A = np.zeros((n, n))
        for u, v, w in self.edges:
            A[u, v] = w
            if not self.directed:
                A[v, u] = w
        return A

# Create graph from edge list
g = EdgeList()
edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 2)]

for u, v in edges:
    g.add_edge(u, v)

print(f"Edge list: {[(u, v) for u, v, _ in g.edges]}")
print(f"Vertices: {sorted(g.vertices())}")
print(f"Neighbors of 0: {g.get_neighbors(0)}")
print(f"\nAdjacency matrix from edge list:\n{g.to_adjacency_matrix().astype(int)}")

---
## Example 5: Sparse Representations - COO Format

The **COO (Coordinate List)** format uses parallel arrays:
- `edge_index[0]` = source nodes
- `edge_index[1]` = target nodes

This is the standard format in **PyTorch Geometric** for GNNs.

In [ ]:
class COOGraph:
    """Graph in COO format (PyTorch Geometric style)."""
    
    def __init__(self, num_nodes: int):
        self.num_nodes = num_nodes
        self.edge_index = [[], []]  # [sources, targets]
        self.edge_attr = []  # Optional edge attributes
    
    def add_edge(self, u: int, v: int, attr: Any = None):
        self.edge_index[0].append(u)
        self.edge_index[1].append(v)
        if attr is not None:
            self.edge_attr.append(attr)
    
    def add_undirected_edge(self, u: int, v: int, attr: Any = None):
        self.add_edge(u, v, attr)
        self.add_edge(v, u, attr)
    
    def get_edge_index(self) -> np.ndarray:
        return np.array(self.edge_index)
    
    def num_edges(self) -> int:
        return len(self.edge_index[0])
    
    def to_adjacency_matrix(self) -> np.ndarray:
        A = np.zeros((self.num_nodes, self.num_nodes))
        for i in range(self.num_edges()):
            u, v = self.edge_index[0][i], self.edge_index[1][i]
            A[u, v] = 1
        return A

# Create graph
g = COOGraph(num_nodes=4)
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
for u, v in edges:
    g.add_undirected_edge(u, v)

edge_index = g.get_edge_index()
print(f"Edge index (COO format):")
print(f"  Source: {edge_index[0].tolist()}")
print(f"  Target: {edge_index[1].tolist()}")
print(f"\nNumber of edges: {g.num_edges()}")
print(f"Adjacency matrix:\n{g.to_adjacency_matrix().astype(int)}")

# Message passing example (GNN style)
print("\n--- Message Passing Example ---")
node_features = np.array([[1.0], [2.0], [3.0], [4.0]])  # 4 nodes, 1 feature
print(f"Node features: {node_features.flatten()}")

# Sum aggregation
aggregated = np.zeros_like(node_features)
for i in range(g.num_edges()):
    src, dst = edge_index[0, i], edge_index[1, i]
    aggregated[dst] += node_features[src]

print(f"After sum aggregation: {aggregated.flatten()}")

---
## Example 6: CSR/CSC Formats

**CSR (Compressed Sparse Row)** format is optimized for row-wise operations:
- `row_ptr[i]`: start index for row i's edges in col_idx
- `col_idx`: column indices of non-zero entries
- `data`: values of non-zero entries

**CSC (Compressed Sparse Column)** is the column-wise version.

CSR is ideal for sparse matrix-vector multiplication (SpMV).

In [ ]:
class CSRGraph:
    """Graph in CSR format."""
    
    def __init__(self, n: int, edges: List[Tuple[int, int]], 
                 weights: Optional[List[float]] = None):
        # Sort edges by source
        if weights is None:
            weights = [1.0] * len(edges)
        
        edge_data = sorted(zip(edges, weights), key=lambda x: x[0][0])
        
        self.n = n
        self.row_ptr = [0] * (n + 1)
        self.col_idx = []
        self.data = []
        
        for (u, v), w in edge_data:
            self.row_ptr[u + 1] += 1
            self.col_idx.append(v)
            self.data.append(w)
        
        # Cumulative sum for row_ptr
        for i in range(1, n + 1):
            self.row_ptr[i] += self.row_ptr[i - 1]
        
        self.row_ptr = np.array(self.row_ptr)
        self.col_idx = np.array(self.col_idx)
        self.data = np.array(self.data)
    
    def get_neighbors(self, v: int) -> np.ndarray:
        start, end = self.row_ptr[v], self.row_ptr[v + 1]
        return self.col_idx[start:end]
    
    def degree(self, v: int) -> int:
        return self.row_ptr[v + 1] - self.row_ptr[v]
    
    def spmv(self, x: np.ndarray) -> np.ndarray:
        """Sparse matrix-vector multiply: A @ x"""
        result = np.zeros(self.n)
        for i in range(self.n):
            start, end = self.row_ptr[i], self.row_ptr[i + 1]
            for j in range(start, end):
                result[i] += self.data[j] * x[self.col_idx[j]]
        return result
    
    def to_dense(self) -> np.ndarray:
        A = np.zeros((self.n, self.n))
        for i in range(self.n):
            start, end = self.row_ptr[i], self.row_ptr[i + 1]
            for j in range(start, end):
                A[i, self.col_idx[j]] = self.data[j]
        return A

# Create symmetric adjacency (both directions for undirected)
edges = [(0, 1), (0, 3), (1, 0), (1, 2), (2, 1), (2, 3), (3, 0), (3, 2)]
g = CSRGraph(4, edges)

print("CSR representation:")
print(f"  row_ptr: {g.row_ptr.tolist()}")
print(f"  col_idx: {g.col_idx.tolist()}")
print(f"  data:    {g.data.tolist()}")

print(f"\nNeighbors of vertex 0: {g.get_neighbors(0).tolist()}")
print(f"Neighbors of vertex 2: {g.get_neighbors(2).tolist()}")
print(f"Degree of vertex 1: {g.degree(1)}")

# Sparse matrix-vector multiply
x = np.array([1.0, 2.0, 3.0, 4.0])
Ax = g.spmv(x)
print(f"\nSparse matrix-vector multiply:")
print(f"  x = {x}")
print(f"  A @ x = {Ax}")
print(f"  Dense verification: {g.to_dense() @ x}")

---
## Example 7: Memory Comparison

Different representations have vastly different memory requirements. For a graph with $n$ nodes and $m$ edges:

| Format | Memory Complexity | Best For |
|--------|-------------------|----------|
| Dense Matrix | $O(n^2)$ | Small, dense graphs |
| Adjacency List | $O(n + m)$ | General use |
| Edge List | $O(m)$ | I/O operations |
| COO | $O(m)$ | GNN frameworks |
| CSR | $O(n + m)$ | Efficient SpMV |

In [ ]:
def analyze_memory(n, m, density=None):
    """Analyze memory usage for graph with n nodes and m edges."""
    if density is not None:
        m = int(density * n * n)
    
    print(f"\nGraph: {n:,} nodes, {m:,} edges, density={m/(n*n):.6f}")
    
    # Dense adjacency matrix (float64)
    dense_bytes = n * n * 8
    print(f"  Dense matrix:      {dense_bytes:>15,} bytes ({dense_bytes/1e6:>8.2f} MB)")
    
    # Adjacency list (approximate)
    adj_list_bytes = n * 56 + m * 2 * 16  # dict overhead + entries
    print(f"  Adjacency list:    {adj_list_bytes:>15,} bytes ({adj_list_bytes/1e6:>8.2f} MB)")
    
    # Edge list
    edge_list_bytes = m * 2 * 8
    print(f"  Edge list:         {edge_list_bytes:>15,} bytes ({edge_list_bytes/1e6:>8.2f} MB)")
    
    # COO
    coo_bytes = m * 2 * 8 + m * 8  # row, col, data arrays
    print(f"  COO:               {coo_bytes:>15,} bytes ({coo_bytes/1e6:>8.2f} MB)")
    
    # CSR
    csr_bytes = (n + 1) * 8 + m * 8 + m * 8  # indptr, indices, data
    print(f"  CSR:               {csr_bytes:>15,} bytes ({csr_bytes/1e6:>8.2f} MB)")
    
    # Savings
    print(f"  Memory savings (CSR vs Dense): {dense_bytes/csr_bytes:.1f}x")

print("=" * 60)
print("Memory Comparison Across Different Graph Sizes")
print("=" * 60)

# Small dense graph
print("\n--- Small Dense Graph ---")
analyze_memory(n=100, m=5000, density=0.5)

# Medium sparse graph (social network like)
print("\n--- Medium Sparse Graph (Social Network) ---")
analyze_memory(n=10000, m=100000)  # Avg degree 20

# Large sparse graph
print("\n--- Large Sparse Graph ---")
analyze_memory(n=1000000, m=10000000)  # 1M nodes, 10M edges

print("\n" + "=" * 60)
print("Conclusion: Use sparse formats for real-world graphs!")
print("Sparse formats provide 100-10000x memory savings.")

---
## Example 8: Conversion Between Formats

Often you need to convert between different graph representations. Here we demonstrate common conversion operations.

In [ ]:
def edge_list_to_adj_list(edges, directed=False):
    """Convert edge list to adjacency list."""
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        if not directed:
            adj[v].append(u)
    return dict(adj)

def adj_list_to_matrix(adj_list, n):
    """Convert adjacency list to matrix."""
    A = np.zeros((n, n))
    for u, neighbors in adj_list.items():
        for v in neighbors:
            A[u, v] = 1
    return A

def matrix_to_edge_list(A):
    """Convert matrix to edge list (upper triangle for undirected)."""
    edges = []
    n = len(A)
    for i in range(n):
        for j in range(i + 1, n):
            if A[i, j] != 0:
                edges.append((i, j))
    return edges

def edge_list_to_coo(edges, n):
    """Convert edge list to COO format."""
    row = [u for u, v in edges]
    col = [v for u, v in edges]
    # Add reverse for undirected
    row_sym = row + col
    col_sym = col + row
    return np.array([row_sym, col_sym])

def coo_to_csr(edge_index, n):
    """Convert COO to CSR using scipy."""
    data = np.ones(edge_index.shape[1])
    return sparse.coo_matrix(
        (data, (edge_index[0], edge_index[1])),
        shape=(n, n)
    ).tocsr()

# Demonstration
edges = [(0, 1), (0, 2), (1, 2), (2, 3)]
n = 4

print(f"Original edge list: {edges}")

# Convert to adjacency list
adj_list = edge_list_to_adj_list(edges)
print(f"\n→ Adjacency list: {dict(adj_list)}")

# Convert to matrix
A = adj_list_to_matrix(adj_list, n)
print(f"\n→ Adjacency matrix:\n{A.astype(int)}")

# Back to edge list
edges_back = matrix_to_edge_list(A)
print(f"\n→ Back to edge list: {edges_back}")

# To COO (PyTorch style)
coo = edge_list_to_coo(edges, n)
print(f"\n→ COO edge_index:\n{coo}")

# To CSR
csr = coo_to_csr(coo, n)
print(f"\n→ CSR format:")
print(f"   indptr: {csr.indptr}")
print(f"   indices: {csr.indices}")

---
## Example 9: Weighted Graph Representations

Weighted graphs store edge weights in addition to connectivity. All major formats support weights:
- **Adjacency Matrix**: $A[i,j] = w_{ij}$
- **Adjacency List**: Store (neighbor, weight) tuples
- **COO/CSR**: Use `data` array for weights

In [ ]:
# Create weighted graph using NetworkX
G = nx.Graph()
weighted_edges = [
    (0, 1, 2.5), (0, 2, 1.0), (1, 2, 3.0),
    (1, 3, 0.5), (2, 3, 2.0), (2, 4, 1.5)
]
G.add_weighted_edges_from(weighted_edges)

print("Weighted Graph (NetworkX)")
print("=" * 40)

# Adjacency matrix with weights
A = nx.to_numpy_array(G)
print(f"Weighted adjacency matrix:\n{A}")

# Adjacency list with weights
print(f"\nWeighted adjacency list:")
for node in G.nodes():
    neighbors = [(v, G[node][v]['weight']) for v in G.neighbors(node)]
    print(f"  {node}: {neighbors}")

# Scipy sparse with weights
A_sparse = nx.to_scipy_sparse_array(G, format='csr')
print(f"\nCSR sparse format:")
print(f"  indptr: {A_sparse.indptr}")
print(f"  indices: {A_sparse.indices}")
print(f"  data (weights): {A_sparse.data}")

# Shortest path using weights
path = nx.dijkstra_path(G, 0, 4, weight='weight')
length = nx.dijkstra_path_length(G, 0, 4, weight='weight')
print(f"\nShortest path from 0 to 4: {path}")
print(f"Path length: {length}")

---
## Example 10: Directed Graph Representations

**Directed graphs (digraphs)** have edges with direction. Key differences:
- Adjacency matrix is typically asymmetric: $A[i,j] \neq A[j,i]$
- In-degree and out-degree are different
- COO format naturally supports direction

In [ ]:
# Create directed graph
G = nx.DiGraph()
edges = [(0, 1), (0, 2), (1, 2), (2, 0), (2, 3), (3, 1)]
G.add_edges_from(edges)

print("Directed Graph")
print("=" * 40)

# Adjacency matrix (note: not symmetric)
A = nx.to_numpy_array(G)
print(f"Adjacency matrix (row=source, col=target):\n{A.astype(int)}")
print(f"\nIs symmetric: {np.allclose(A, A.T)}")

# In-degree and out-degree
print(f"\nDegree information:")
for node in G.nodes():
    print(f"  Node {node}: in-degree={G.in_degree(node)}, out-degree={G.out_degree(node)}")

# Successors (outgoing) vs predecessors (incoming)
print(f"\nNode 2:")
print(f"  Successors (outgoing): {list(G.successors(2))}")
print(f"  Predecessors (incoming): {list(G.predecessors(2))}")

# COO format for directed graph
edge_index = np.array([[u, v] for u, v in G.edges()]).T
print(f"\nCOO edge_index:")
print(f"  Source: {edge_index[0].tolist()}")
print(f"  Target: {edge_index[1].tolist()}")

# Strongly connected components
sccs = list(nx.strongly_connected_components(G))
print(f"\nStrongly connected components: {sccs}")

---
## Example 11: Multigraph Representations

**Multigraphs** allow multiple edges between the same pair of vertices. These are common in:
- Knowledge graphs (multiple relation types)
- Transportation networks (multiple routes)
- Molecular graphs (multiple bond types)

In [ ]:
# Create multigraph with NetworkX
G = nx.MultiGraph()

# Add multiple edges between same nodes with different attributes
G.add_edge(0, 1, weight=1.0, relation='friend')
G.add_edge(0, 1, weight=2.0, relation='colleague')
G.add_edge(1, 2, weight=1.5, relation='friend')
G.add_edge(1, 2, weight=0.5, relation='family')
G.add_edge(0, 2, weight=1.0, relation='colleague')

print("Multigraph (Multiple edges between nodes)")
print("=" * 50)

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

# View all edges with attributes
print(f"\nAll edges:")
for u, v, key, data in G.edges(keys=True, data=True):
    print(f"  ({u}, {v}, key={key}): {data}")

# Edge list representation (includes edge keys)
print(f"\nMultigraph edge list:")
edge_list = [(u, v, key, data['relation']) for u, v, key, data in G.edges(keys=True, data=True)]
for edge in edge_list:
    print(f"  {edge}")

# COO with edge types
class MultiEdgeCOO:
    """COO format with edge types for multigraphs."""
    def __init__(self):
        self.edge_index = [[], []]
        self.edge_type = []
        self.edge_weight = []
    
    def add_edge(self, u, v, edge_type, weight=1.0):
        self.edge_index[0].append(u)
        self.edge_index[1].append(v)
        self.edge_type.append(edge_type)
        self.edge_weight.append(weight)
    
    def get_edge_index(self):
        return np.array(self.edge_index)

# Convert to COO with edge types
coo = MultiEdgeCOO()
relation_map = {'friend': 0, 'colleague': 1, 'family': 2}
for u, v, key, data in G.edges(keys=True, data=True):
    coo.add_edge(u, v, relation_map[data['relation']], data['weight'])
    coo.add_edge(v, u, relation_map[data['relation']], data['weight'])  # Undirected

print(f"\nCOO with edge types:")
print(f"  edge_index shape: {coo.get_edge_index().shape}")
print(f"  edge_types: {coo.edge_type}")
print(f"  edge_weights: {coo.edge_weight}")

---
## Example 12: Choosing the Right Representation

Choosing the best representation depends on your use case:

| Use Case | Recommended Format | Reason |
|----------|-------------------|--------|
| Small dense graphs | Adjacency Matrix | Fast operations, $O(1)$ edge lookup |
| Large sparse graphs | CSR/CSC | Memory efficient, fast SpMV |
| GNN frameworks | COO (edge_index) | PyTorch Geometric standard |
| Dynamic graphs | Adjacency List | Easy add/remove |
| File I/O | Edge List | Simple format |
| Spectral methods | Scipy Sparse | Eigenvalue computation |

In [ ]:
import time

def benchmark_representations(n, m, num_ops=1000):
    """Compare operation speed across representations."""
    print(f"\nBenchmark: {n} nodes, {m} edges")
    print("=" * 50)
    
    # Generate random graph
    np.random.seed(42)
    edges = set()
    while len(edges) < m:
        u, v = np.random.randint(0, n, 2)
        if u != v:
            edges.add((u, v))
    edges = list(edges)
    
    # Dense matrix
    A_dense = np.zeros((n, n))
    for u, v in edges:
        A_dense[u, v] = 1
        A_dense[v, u] = 1
    
    # Scipy CSR
    row = [u for u, v in edges] + [v for u, v in edges]
    col = [v for u, v in edges] + [u for u, v in edges]
    A_csr = sparse.csr_matrix((np.ones(len(row)), (row, col)), shape=(n, n))
    
    # Adjacency list
    adj_list = defaultdict(set)
    for u, v in edges:
        adj_list[u].add(v)
        adj_list[v].add(u)
    
    # Test vector for SpMV
    x = np.random.rand(n)
    
    # Benchmark SpMV
    start = time.time()
    for _ in range(num_ops):
        _ = A_dense @ x
    dense_time = time.time() - start
    
    start = time.time()
    for _ in range(num_ops):
        _ = A_csr @ x
    sparse_time = time.time() - start
    
    print(f"\nMatrix-Vector Multiply ({num_ops} ops):")
    print(f"  Dense:  {dense_time:.4f}s")
    print(f"  Sparse: {sparse_time:.4f}s")
    print(f"  Speedup: {dense_time/sparse_time:.2f}x")
    
    # Benchmark edge lookup
    test_pairs = [(np.random.randint(0, n), np.random.randint(0, n)) for _ in range(num_ops)]
    
    start = time.time()
    for u, v in test_pairs:
        _ = A_dense[u, v] != 0
    dense_lookup = time.time() - start
    
    start = time.time()
    for u, v in test_pairs:
        _ = v in adj_list[u]
    adj_lookup = time.time() - start
    
    print(f"\nEdge Lookup ({num_ops} ops):")
    print(f"  Dense matrix: {dense_lookup:.4f}s")
    print(f"  Adjacency list (set): {adj_lookup:.4f}s")
    
    # Memory comparison
    dense_mem = A_dense.nbytes
    sparse_mem = A_csr.data.nbytes + A_csr.indices.nbytes + A_csr.indptr.nbytes
    
    print(f"\nMemory:")
    print(f"  Dense:  {dense_mem/1e6:.2f} MB")
    print(f"  Sparse: {sparse_mem/1e6:.2f} MB")
    print(f"  Savings: {dense_mem/sparse_mem:.1f}x")

# Run benchmarks
benchmark_representations(n=500, m=5000, num_ops=100)

print("\n" + "=" * 50)
print("Recommendations:")
print("=" * 50)
print("""
1. For GNNs: Use COO (edge_index) format
   - Standard in PyTorch Geometric and DGL
   - Easy message passing

2. For large sparse graphs: Use CSR
   - 100-1000x memory savings
   - Fast SpMV for spectral methods

3. For dynamic graphs: Use adjacency list with sets
   - O(1) edge add/remove/lookup

4. For dense matrix operations: Use NumPy arrays
   - Matrix powers, eigenvalues
   - Only for small graphs (< 10K nodes)
""")